# Ddri 대여소 좌표 -> 행정동 매핑 노트북

목적:
- 강남구 대여소 좌표를 행정동으로 역변환한다.
- `서울시_행정동코드_매핑표.csv`와 연결해 `행정동코드`를 붙인다.
- 이후 생활인구 결합에 사용할 수 있는 대여소-행정동 매핑 테이블을 만든다.

사용 API:
- Kakao Local API `coord2regioncode`

출력:
- `cheng80/ddri_station_to_dong_mapping_2025.csv`

주의:
- 이 노트북은 API 키가 필요하다.
- 키는 코드에 직접 쓰지 않고 환경변수에서 읽는다.

## 사전 준비

환경변수 설정 예시:

```bash
export KAKAO_REST_API_KEY='여기에_카카오_REST_API_KEY'
```

이후 노트북을 실행하면 `os.environ['KAKAO_REST_API_KEY']`에서 키를 읽는다.

In [1]:
from pathlib import Path
import os
import time

import pandas as pd
import requests

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
RAW_DIR = BASE_DIR / '3조 공유폴더'
OUTPUT_DIR = BASE_DIR / 'cheng80'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KAKAO_REST_API_KEY = os.environ.get('KAKAO_REST_API_KEY')

if not KAKAO_REST_API_KEY:
    raise ValueError('KAKAO_REST_API_KEY 환경변수가 설정되지 않았습니다.')

BASE_DIR, OUTPUT_DIR

(PosixPath('/Users/cheng80/Desktop/ddri_work'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/cheng80'))

## 입력 데이터 로드

이번 시도에서는 우선 2025 대여소 정보를 기준으로 대여소-행정동 매핑을 만든다.

입력:
- `강남구 대여소 정보 (2023~2025)/2025_강남구_대여소.csv`
- `서울시_행정동코드_매핑표.csv`

In [2]:
station_df = pd.read_csv(RAW_DIR / '강남구 대여소 정보 (2023~2025)' / '2025_강남구_대여소.csv')
dong_map_df = pd.read_csv(RAW_DIR / '서울시_행정동코드_매핑표.csv')

station_df['대여소번호'] = pd.to_numeric(station_df['대여소번호'], errors='coerce').astype('Int64')
station_df['위도'] = pd.to_numeric(station_df['위도'], errors='coerce')
station_df['경도'] = pd.to_numeric(station_df['경도'], errors='coerce')
station_df = station_df.dropna(subset=['대여소번호', '위도', '경도']).copy()
station_df['대여소번호'] = station_df['대여소번호'].astype(int)

dong_map_df['행정동코드'] = dong_map_df['행정동코드'].astype(str)
gangnam_dong_map_df = dong_map_df[dong_map_df['행정동코드'].str.startswith('11680')].copy()

print('station_rows =', len(station_df))
print('gangnam_dong_rows =', len(gangnam_dong_map_df))
station_df[['대여소번호', '대여소명', '주소', '위도', '경도']].head()

station_rows = 179
gangnam_dong_rows = 22


,대여소번호,대여소명,주소,위도,경도
0,2301,현대고등학교 건너편,서울특별시 강남구 압구정로 134,37.524071,127.021790
1,2302,교보타워 버스정류장(신논현역 3번출구 후면),서울특별시 강남구 봉은사로 지하 102,37.505581,127.024277
2,2303,논현역 10번출구,서울특별시 강남구 학동로 지하 102,37.511372,127.021461
3,2304,대현그린타워,서울특별시 강남구 언주로 626,37.511627,127.035652
4,2305,MCM 본사 직영점 앞,서울특별시 강남구 언주로 734,37.520641,127.034508


## Kakao Local API 호출 함수

카카오 `coord2regioncode`는 좌표를 행정구역 정보로 변환한다.
여기서는 `행정동(region_type='H')`를 우선적으로 사용한다.

In [3]:
def fetch_kakao_region(lon: float, lat: float) -> dict:
    url = 'https://dapi.kakao.com/v2/local/geo/coord2regioncode.json'
    headers = {'Authorization': f'KakaoAK {KAKAO_REST_API_KEY}'}
    params = {'x': lon, 'y': lat}
    resp = requests.get(url, headers=headers, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    docs = data.get('documents', [])
    if not docs:
        return {}

    h_docs = [d for d in docs if d.get('region_type') == 'H']
    target = h_docs[0] if h_docs else docs[0]
    return {
        'api_region_type': target.get('region_type'),
        'api_region_1depth_name': target.get('region_1depth_name'),
        'api_region_2depth_name': target.get('region_2depth_name'),
        'api_region_3depth_name': target.get('region_3depth_name'),
        'api_region_4depth_name': target.get('region_4depth_name'),
        'api_code': str(target.get('code')) if target.get('code') is not None else None,
    }


## 단건 테스트

전체 호출 전에 샘플 1건이 정상적으로 동작하는지 확인한다.

In [4]:
sample = station_df.iloc[0]
sample_result = fetch_kakao_region(sample['경도'], sample['위도'])
sample[['대여소번호', '대여소명', '주소', '위도', '경도']], sample_result

(대여소번호                  2301
 대여소명             현대고등학교 건너편
 주소       서울특별시 강남구 압구정로 134
 위도                37.524071
 경도                127.02179
 Name: 0, dtype: object,
 {'api_region_type': 'H',
  'api_region_1depth_name': '서울특별시',
  'api_region_2depth_name': '강남구',
  'api_region_3depth_name': '신사동',
  'api_region_4depth_name': '',
  'api_code': '1168051000'})

## 전체 대여소 좌표 -> 행정동 변환

API 호출량을 고려해 아주 짧은 sleep을 둔다.
필요하면 추후 캐시 파일을 두는 방식으로 고도화할 수 있다.

In [5]:
rows = []
for idx, row in station_df.iterrows():
    region = fetch_kakao_region(row['경도'], row['위도'])
    rows.append({
        '대여소번호': row['대여소번호'],
        '대여소명': row['대여소명'],
        '주소': row['주소'],
        '위도': row['위도'],
        '경도': row['경도'],
        **region,
    })
    if (idx + 1) % 20 == 0:
        print(f'processed {idx + 1} / {len(station_df)}')
    time.sleep(0.05)

region_df = pd.DataFrame(rows)
region_df.head()

processed 20 / 179


processed 40 / 179


processed 60 / 179


processed 80 / 179


processed 100 / 179


processed 120 / 179


processed 140 / 179


processed 160 / 179


,대여소번호,대여소명,주소,위도,경도,api_region_type,api_region_1depth_name,api_region_2depth_name,api_region_3depth_name,api_region_4depth_name,api_code
0,2301,현대고등학교 건너편,서울특별시 강남구 압구정로 134,37.524071,127.021790,H,서울특별시,강남구,신사동,,1168051000
1,2302,교보타워 버스정류장(신논현역 3번출구 후면),서울특별시 강남구 봉은사로 지하 102,37.505581,127.024277,H,서울특별시,강남구,논현1동,,1168052100
2,2303,논현역 10번출구,서울특별시 강남구 학동로 지하 102,37.511372,127.021461,H,서울특별시,강남구,논현1동,,1168052100
3,2304,대현그린타워,서울특별시 강남구 언주로 626,37.511627,127.035652,H,서울특별시,강남구,논현2동,,1168053100
4,2305,MCM 본사 직영점 앞,서울특별시 강남구 언주로 734,37.520641,127.034508,H,서울특별시,강남구,논현2동,,1168053100


## 행정동코드 매핑표와 결합

카카오 응답의 `region_3depth_name`을 `행정동명`과 맞춰 보고,
가능하면 `행정동코드`를 붙인다.

주의:
- API 응답 명칭과 매핑표 명칭이 완전히 같지 않을 수 있다.
- 이 경우 일부는 수동 검수 또는 후처리 규칙이 필요할 수 있다.

In [6]:
mapping_df = gangnam_dong_map_df.rename(columns={'행정동명': 'mapped_dong_name', '행정동코드': 'mapped_dong_code'})

# 매핑표에 누락된 동은 별도 보정 테이블로 관리한다.
# 개포3동은 카카오 응답으로 확인되지만 현재 공유받은 매핑표에는 빠져 있어 수동 보정한다.
manual_override_df = pd.DataFrame(
    [
        {'mapped_dong_name': '개포3동', 'mapped_dong_code': '11680675'},
    ]
)
mapping_df = pd.concat([mapping_df, manual_override_df], ignore_index=True).drop_duplicates(
    subset=['mapped_dong_name'], keep='first'
)

station_dong_df = region_df.merge(
    mapping_df,
    left_on='api_region_3depth_name',
    right_on='mapped_dong_name',
    how='left'
)

station_dong_df[['대여소번호', '대여소명', 'api_region_3depth_name', 'mapped_dong_name', 'mapped_dong_code']].head(20)

,대여소번호,대여소명,api_region_3depth_name,mapped_dong_name,mapped_dong_code
0,2301,현대고등학교 건너편,신사동,신사동,11680510
1,2302,교보타워 버스정류장(신논현역 3번출구 후면),논현1동,논현1동,11680521
2,2303,논현역 10번출구,논현1동,논현1동,11680521
3,2304,대현그린타워,논현2동,논현2동,11680531
4,2305,MCM 본사 직영점 앞,논현2동,논현2동,11680531
5,2306,압구정역 2번 출구 옆,압구정동,압구정동,11680545
6,2307,압구정 한양 3차 아파트,압구정동,압구정동,11680545
7,2308,압구정파출소 앞,압구정동,압구정동,11680545
8,2309,청담역(우리들병원 앞),청담동,청담동,11680565
9,2310,청담동 맥도날드 옆(위치),청담동,청담동,11680565


## 매핑 성공률 점검

여기서 `mapped_dong_code`가 비어 있지 않은 비율을 확인한다.
성공률이 높으면 생활인구 결합 기반으로 사용할 수 있다.

In [7]:
mapped_count = station_dong_df['mapped_dong_code'].notna().sum()
total_count = len(station_dong_df)
print('mapped_count =', mapped_count)
print('total_count =', total_count)
print('mapped_ratio =', round(mapped_count / total_count, 4) if total_count else None)

station_dong_df[station_dong_df['mapped_dong_code'].isna()][['대여소번호', '대여소명', '주소', 'api_region_3depth_name']].head(20)

mapped_count = 179
total_count = 179
mapped_ratio = 1.0


,대여소번호,대여소명,주소,api_region_3depth_name


## CSV 저장

최종 출력은 검토용으로 `cheng80/`에 저장한다.

In [8]:
output_path = OUTPUT_DIR / 'ddri_station_to_dong_mapping_2025.csv'
station_dong_df.to_csv(output_path, index=False)
print('saved:', output_path)

saved: /Users/cheng80/Desktop/ddri_work/cheng80/ddri_station_to_dong_mapping_2025.csv


## 다음 단계

이 매핑이 안정적으로 붙으면 다음이 가능하다.

1. 대여소별 `행정동코드` 확보
2. 생활인구 CSV의 `행정동코드`와 연결
3. 아침/점심/저녁 시간대 생활인구를 군집화 피처 후보로 생성

즉, 이 노트북은 생활인구 결합의 중간 단계다.